In [46]:
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error,mean_absolute_error
from tensorflow.keras.models import Sequential,load_model
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [47]:
df=pd.read_csv('household_power_consumption.csv',low_memory=False)
print(df.head())
print("\nDataset loaded successfully!")

         Date      Time Global_active_power Global_reactive_power Voltage  \
0  16/12/2006  17:24:00               4.216                 0.418  234.84   
1  16/12/2006  17:25:00                5.36                 0.436  233.63   
2  16/12/2006  17:26:00               5.374                 0.498  233.29   
3  16/12/2006  17:27:00               5.388                 0.502  233.74   
4  16/12/2006  17:28:00               3.666                 0.528  235.68   

  Global_intensity Sub_metering_1 Sub_metering_2  Sub_metering_3  
0             18.4              0              1            17.0  
1               23              0              1            16.0  
2               23              0              2            17.0  
3               23              0              1            17.0  
4             15.8              0              1            17.0  

Dataset loaded successfully!


In [48]:
print("\nShape of the dataset:", df.shape)
print("\nFirst 5 rows of the dataset:")
print(df.head())
print("\nData types of the Information:")
print(df.info())
print("\nMissing values in the dataset:")
print(df.isnull().sum())


Shape of the dataset: (1048575, 9)

First 5 rows of the dataset:
         Date      Time Global_active_power Global_reactive_power Voltage  \
0  16/12/2006  17:24:00               4.216                 0.418  234.84   
1  16/12/2006  17:25:00                5.36                 0.436  233.63   
2  16/12/2006  17:26:00               5.374                 0.498  233.29   
3  16/12/2006  17:27:00               5.388                 0.502  233.74   
4  16/12/2006  17:28:00               3.666                 0.528  235.68   

  Global_intensity Sub_metering_1 Sub_metering_2  Sub_metering_3  
0             18.4              0              1            17.0  
1               23              0              1            16.0  
2               23              0              2            17.0  
3               23              0              1            17.0  
4             15.8              0              1            17.0  

Data types of the Information:
<class 'pandas.core.frame.DataFrame'>

In [49]:
import numpy as np
df.replace("?",np.nan,inplace=True)

In [50]:
import pandas as pd

numeric_columns = [
    'Global_active_power',
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3'
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing values using forward fill
df.ffill(inplace=True)

In [51]:
df['Datetime']=pd.to_datetime(
    df['Date']+' '+df['Time'],
    dayfirst=True
)

In [52]:
df=df.sort_values("Datetime")
print(df.columns)

Index(['Date', 'Time', 'Global_active_power', 'Global_reactive_power',
       'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2',
       'Sub_metering_3', 'Datetime'],
      dtype='object')


In [53]:
df.set_index('Datetime', inplace=True)
print("\nData After Cleaning:")
print(df.head())


Data After Cleaning:
                           Date      Time  Global_active_power  \
Datetime                                                         
2006-12-16 17:24:00  16/12/2006  17:24:00                4.216   
2006-12-16 17:25:00  16/12/2006  17:25:00                5.360   
2006-12-16 17:26:00  16/12/2006  17:26:00                5.374   
2006-12-16 17:27:00  16/12/2006  17:27:00                5.388   
2006-12-16 17:28:00  16/12/2006  17:28:00                3.666   

                     Global_reactive_power  Voltage  Global_intensity  \
Datetime                                                                
2006-12-16 17:24:00                  0.418   234.84              18.4   
2006-12-16 17:25:00                  0.436   233.63              23.0   
2006-12-16 17:26:00                  0.498   233.29              23.0   
2006-12-16 17:27:00                  0.502   233.74              23.0   
2006-12-16 17:28:00                  0.528   235.68              15.8   

   

In [54]:
features=[
    'Global_active_power',
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3'
]
data=df[features]
print("\nSelected features for modeling:")
print(data.head())


Selected features for modeling:
                     Global_active_power  Global_reactive_power  Voltage  \
Datetime                                                                   
2006-12-16 17:24:00                4.216                  0.418   234.84   
2006-12-16 17:25:00                5.360                  0.436   233.63   
2006-12-16 17:26:00                5.374                  0.498   233.29   
2006-12-16 17:27:00                5.388                  0.502   233.74   
2006-12-16 17:28:00                3.666                  0.528   235.68   

                     Global_intensity  Sub_metering_1  Sub_metering_2  \
Datetime                                                                
2006-12-16 17:24:00              18.4             0.0             1.0   
2006-12-16 17:25:00              23.0             0.0             1.0   
2006-12-16 17:26:00              23.0             0.0             2.0   
2006-12-16 17:27:00              23.0             0.0             1.0

In [55]:
scalar=MinMaxScaler()
scaled_data=scalar.fit_transform(data)
print("\nScaled Data Shape:")
print(scaled_data.shape)


Scaled Data Shape:
(1048575, 7)


In [56]:
sequence_length=30
x=[]
y=[]
for i in range(sequence_length, len(scaled_data)):
  x.append(scaled_data[i-sequence_length:i])
  y.append(scaled_data[i,0])

In [57]:
x=np.array(x)
y=np.array(y)
print("x Shape:",x.shape)
print("y Shape:", y.shape)

x Shape: (1048545, 30, 7)
y Shape: (1048545,)


In [58]:
train_size=int(len(scaled_data)*0.8)
x_train=x[:train_size]
x_test=x[train_size:]
y_train=y[:train_size]
y_test=y[train_size:]
print("\nTraining Samples:" + str(len(x_train)))
print("\nTesting Samples:" + str(len(x_test)))

print("\nX_train:", x_train.shape)
print("\nX_test:", x_test.shape)
print("\nY_train:", y_train.shape)
print("\nY_test:", y_test.shape)


Training Samples:838860

Testing Samples:209685

X_train: (838860, 30, 7)

X_test: (209685, 30, 7)

Y_train: (838860,)

Y_test: (209685,)


In [59]:
print("\nOne Sequence shape:")
print(x_train[0].shape)
print("\nTarget Value:")
print(y_train[0])


One Sequence shape:
(30, 7)

Target Value:
0.24957523126297906


In [60]:
model=Sequential()
model.add(SimpleRNN(units=64,activation="tanh", return_sequences=True, input_shape=(x_train.shape[1], x_train.shape[2])))

d:\cancer-mechlearn\venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [61]:
model.add(SimpleRNN(units=32, activation="tanh"))

In [62]:
model.add(Dropout(0.2))

In [63]:
model.add(Dense(1))

In [64]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_4 (SimpleRNN)        │ (None, 30, 64)         │         4,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_5 (SimpleRNN)        │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,745 (30.25 KB)

 Trainable params: 7,745 (30.25 KB)

 Non-trainable params: 0 (0.00 B)

In [65]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
print("\nModel compiled successfully!")


Model compiled successfully!


In [68]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)
history=model.fit(x_train,y_train,epochs=20,batch_size=64,validation_data=(x_test, y_test),callbacks=[early_stop],verbose=1)

Epoch 1/20
13108/13108 ━━━━━━━━━━━━━━━━━━━━ 89s 7ms/step - loss: 0.0014 - mae: 0.0193 - val_loss: 4.9125e-04 - val_mae: 0.0101
Epoch 2/20
13108/13108 ━━━━━━━━━━━━━━━━━━━━ 112s 9ms/step - loss: 8.3010e-04 - mae: 0.0142 - val_loss: 5.0954e-04 - val_mae: 0.0101
Epoch 3/20
13108/13108 ━━━━━━━━━━━━━━━━━━━━ 116s 9ms/step - loss: 8.2435e-04 - mae: 0.0141 - val_loss: 4.7264e-04 - val_mae: 0.0085
Epoch 4/20
13108/13108 ━━━━━━━━━━━━━━━━━━━━ 123s 9ms/step - loss: 8.1696e-04 - mae: 0.0140 - val_loss: 4.7019e-04 - val_mae: 0.0082
Epoch 5/20
13108/13108 ━━━━━━━━━━━━━━━━━━━━ 124s 9ms/step - loss: 8.1566e-04 - mae: 0.0139 - val_loss: 5.2125e-04 - val_mae: 0.0123
Epoch 6/20
13108/13108 ━━━━━━━━━━━━━━━━━━━━ 132s 10ms/step - loss: 8.1244e-04 - mae: 0.0139 - val_loss: 4.7396e-04 - val_mae: 0.0085
Epoch 7/20
13108/13108 ━━━━━━━━━━━━━━━━━━━━ 129s 10ms/step - loss: 8.1043e-04 - mae: 0.0139 - val_loss: 5.1575e-04 - val_mae: 0.0121
Epoch 8/20
13108/13108 ━━━━━━━━━━━━━━━━━━━━ 135s 10ms/step - loss: 8.1030e-04 -